# 📖 Notebook 3: Real-Time Collaboration via WebSockets

Now that we understand OT and CRDTs, let's build the **real-time experience**. In this notebook, we connect to our live doc server over WebSockets, send edits, and see how multiple users collaborate on the same document.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why WebSockets are used for collaborative editing (not HTTP)
- The connection lifecycle: connect → join document → edit → disconnect
- How cursor presence works (seeing where other users are typing)
- The server-side broadcast pattern for real-time sync

## 🛠️ Setup

Make sure the doc server is running:

```bash
cd system-designs/google-docs
docker-compose up -d
```

Verify the server is up:
```bash
docker logs googledocs-doc-server
```

You should see: `🚀 Google Docs Collaboration Server starting on ws://0.0.0.0:8765`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import asyncio
import json
import websockets
import redis

WS_URL = "ws://localhost:8765"
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
async def test_ws():
    try:
        async with websockets.connect(WS_URL) as ws:
            await ws.send(json.dumps({"type": "connect", "user_id": 1}))
            resp = json.loads(await ws.recv())
            assert resp["type"] == "connected"
            print("✅ WebSocket server is running")
    except Exception as e:
        print(f"❌ WebSocket failed: {e}")
        print("   Run: docker-compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

await test_ws()

✅ Redis connected
✅ WebSocket server is running


## 🤔 Why WebSockets?

In a collaborative editor, the server needs to **push** changes to clients the moment another user types. Regular HTTP can't do this efficiently:

| Approach | How It Works | Problem |
|----------|-------------|----------|
| **HTTP Polling** | Client asks "any changes?" every 100ms | Wasteful — most polls return nothing |
| **Long Polling** | Client waits for a response until something changes | Reconnection overhead, not truly real-time |
| **Server-Sent Events** | Server pushes events to client | One-directional only (server → client) |
| **WebSockets** ✅ | Persistent bi-directional connection | Both sides can send messages anytime |

WebSockets are the standard choice for collaborative editors because:
- **Bi-directional**: client sends edits, server broadcasts to others
- **Low latency**: no connection setup overhead per message
- **Persistent**: stays open for the entire editing session

In [2]:
# Let's compare: How many bytes does it take to send an edit?

# HTTP POST request
http_headers = """POST /api/docs/1/edit HTTP/1.1
Host: localhost:8000
Content-Type: application/json
Authorization: Bearer token123
Content-Length: 56
"""
http_body = json.dumps({"op_type": "insert", "position": 5, "content": "x"})
http_total = len(http_headers) + len(http_body)

# WebSocket message (after connection is established)
ws_message = json.dumps({"type": "edit", "document_id": 1, "op_type": "insert", "position": 5, "content": "x"})
ws_total = len(ws_message) + 6  # ~6 bytes WebSocket frame overhead

print("📊 Bytes Per Edit")
print("=" * 40)
print(f"HTTP POST:  {http_total:>4} bytes  (headers + body)")
print(f"WebSocket:  {ws_total:>4} bytes  (frame + payload)")
print(f"Savings:    {((http_total - ws_total) / http_total * 100):.0f}%")
print()
print("At 5 keystrokes/second for 10 users, that's:")
print(f"  HTTP:      {http_total * 5 * 10:,} bytes/sec")
print(f"  WebSocket: {ws_total * 5 * 10:,} bytes/sec")
print()
print("💡 WebSockets avoid resending headers on every message.")

📊 Bytes Per Edit
HTTP POST:   185 bytes  (headers + body)
WebSocket:    92 bytes  (frame + payload)
Savings:    50%

At 5 keystrokes/second for 10 users, that's:
  HTTP:      9,250 bytes/sec
  WebSocket: 4,600 bytes/sec

💡 WebSockets avoid resending headers on every message.


## 🔌 The Connection Lifecycle

Here's what happens when a user opens a document:

```
1. CONNECT        →  Authenticate with user_id
2. JOIN_DOC       →  Join a specific document for editing
3. RECEIVE STATE  ←  Server sends current document text + who's editing
4. EDIT LOOP      ↔  Send edits, receive others' edits (continuous)
5. DISCONNECT     →  Connection closes, server removes from presence
```

Let's walk through each step.

In [3]:
# Helper class to make WebSocket interactions easier in notebooks

class DocClient:
    """A simple Google Docs client for notebook demos."""
    
    def __init__(self, user_id, name):
        self.user_id = user_id
        self.name = name
        self.ws = None
        self.doc_text = ""
        self.doc_id = None
        self.messages = []  # buffer of received messages
    
    async def connect(self):
        """Step 1: Open WebSocket and authenticate."""
        self.ws = await websockets.connect(WS_URL)
        await self.ws.send(json.dumps({"type": "connect", "user_id": self.user_id}))
        resp = json.loads(await self.ws.recv())
        print(f"  [{self.name}] Connected: {resp}")
    
    async def join_doc(self, doc_id):
        """Step 2: Join a document for editing."""
        self.doc_id = doc_id
        await self.ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        
        # Receive doc_state
        resp = json.loads(await self.ws.recv())
        if resp["type"] == "doc_state":
            self.doc_text = resp["text"]
            print(f"  [{self.name}] Joined doc {doc_id} (version {resp['version']})")
            print(f"  [{self.name}] Document: '{self.doc_text[:80]}...'" if len(self.doc_text) > 80 else f"  [{self.name}] Document: '{self.doc_text}'")
        
        # Receive presence_list
        resp = json.loads(await self.ws.recv())
        if resp["type"] == "presence_list":
            users = resp.get("users", [])
            if users:
                names = [u["name"] for u in users]
                print(f"  [{self.name}] Other editors: {', '.join(names)}")
            else:
                print(f"  [{self.name}] No other editors currently")
    
    async def insert(self, position, content):
        """Send an insert operation."""
        await self.ws.send(json.dumps({
            "type": "edit",
            "document_id": self.doc_id,
            "op_type": "insert",
            "position": position,
            "content": content,
        }))
        # Apply locally
        self.doc_text = self.doc_text[:position] + content + self.doc_text[position:]
        # Read ACK
        resp = json.loads(await self.ws.recv())
        print(f"  [{self.name}] INSERT({position}, '{content}') → ACK")
    
    async def delete(self, position, length=1):
        """Send a delete operation."""
        await self.ws.send(json.dumps({
            "type": "edit",
            "document_id": self.doc_id,
            "op_type": "delete",
            "position": position,
            "length": length,
        }))
        self.doc_text = self.doc_text[:position] + self.doc_text[position + length:]
        resp = json.loads(await self.ws.recv())
        print(f"  [{self.name}] DELETE({position}, {length}) → ACK")
    
    async def update_cursor(self, position):
        """Send a cursor position update."""
        await self.ws.send(json.dumps({
            "type": "cursor_update",
            "document_id": self.doc_id,
            "position": position,
        }))
    
    async def recv_message(self, timeout=0.5):
        """Try to receive a message (with timeout)."""
        try:
            raw = await asyncio.wait_for(self.ws.recv(), timeout=timeout)
            msg = json.loads(raw)
            self.messages.append(msg)
            return msg
        except asyncio.TimeoutError:
            return None
    
    async def recv_all(self, timeout=0.5):
        """Receive all pending messages."""
        messages = []
        while True:
            msg = await self.recv_message(timeout)
            if msg is None:
                break
            messages.append(msg)
        return messages
    
    async def disconnect(self):
        """Close the WebSocket connection."""
        if self.ws:
            await self.ws.close()
            print(f"  [{self.name}] Disconnected")

print("DocClient helper class defined! ✅")

DocClient helper class defined! ✅


In [4]:
# Step 1 & 2: Connect and join a document

alice = DocClient(user_id=1, name="Alice")
await alice.connect()
await alice.join_doc(1)  # Join "Meeting Notes — Q1 Planning"

  [Alice] Connected: {'type': 'connected', 'user_id': 1}
  [Alice] Joined doc 1 (version 1)
  [Alice] Document: 'Meeting Notes — Q1 Planning\n\nAttendees: Alice, Bob, Charlie\n\nAgenda:\n1. Rev...'
  [Alice] No other editors currently


In [5]:
# Step 3: Bob joins the same document — Alice gets a notification!

bob = DocClient(user_id=2, name="Bob")
await bob.connect()
await bob.join_doc(1)  # Join the same document

# Alice should receive a "user_joined" notification
alice_msg = await alice.recv_message(timeout=1)
if alice_msg:
    print(f"\n  Alice received: {alice_msg['type']} → {alice_msg.get('name', '')} joined!")

  [Bob] Connected: {'type': 'connected', 'user_id': 2}
  [Bob] Joined doc 1 (version 1)
  [Bob] Document: 'Meeting Notes — Q1 Planning\n\nAttendees: Alice, Bob, Charlie\n\nAgenda:\n1. Rev...'
  [Bob] Other editors: Alice Johnson

  Alice received: user_joined → Bob Smith joined!


In [6]:
# Step 4: Alice makes an edit — Bob receives it in real-time!

print("Alice types at the end of the document...")
insert_pos = len(alice.doc_text)
await alice.insert(insert_pos, "\n- ACTION: Schedule follow-up meeting")

# Bob should receive the remote operation
bob_msg = await bob.recv_message(timeout=1)
if bob_msg and bob_msg["type"] == "remote_op":
    print(f"\n  Bob received remote_op: {bob_msg['op_type'].upper()} at position {bob_msg['position']}")
    print(f"  Content: '{bob_msg.get('content', '')[:50]}'")
    # Apply to Bob's local copy
    bob.doc_text = bob.doc_text[:bob_msg["position"]] + bob_msg.get("content", "") + bob.doc_text[bob_msg["position"]:]
    print(f"  Bob's document now ends with: '...{bob.doc_text[-50:]}'")

Alice types at the end of the document...
  [Alice] INSERT(250, '
- ACTION: Schedule follow-up meeting') → ACK

  Bob received remote_op: INSERT at position 250
  Content: '
- ACTION: Schedule follow-up meeting'
  Bob's document now ends with: '...d to March 15
- ACTION: Schedule follow-up meeting'


In [7]:
# Step 4 continued: Bob edits too — Alice receives it!

print("Bob types at the end of the document...")
insert_pos = len(bob.doc_text)
await bob.insert(insert_pos, "\n- ACTION: Review budget proposal")

# Alice should receive it
alice_msg = await alice.recv_message(timeout=1)
if alice_msg and alice_msg["type"] == "remote_op":
    alice.doc_text = alice.doc_text[:alice_msg["position"]] + alice_msg.get("content", "") + alice.doc_text[alice_msg["position"]:]
    print(f"\n  Alice received Bob's edit")
    print(f"  Both documents now match: {alice.doc_text == bob.doc_text}")

Bob types at the end of the document...
  [Bob] INSERT(287, '
- ACTION: Review budget proposal') → ACK

  Alice received Bob's edit
  Both documents now match: True


## 👆 Cursor Presence

One of the most important UX features: seeing **where** other users are editing. This is done by broadcasting cursor positions through the server and storing them in Redis.

In [8]:
# Alice moves her cursor to position 50
await alice.update_cursor(50)

# Bob should receive the cursor update
bob_msg = await bob.recv_message(timeout=1)
if bob_msg and bob_msg["type"] == "cursor_update":
    print(f"Bob sees Alice's cursor at position {bob_msg['position']}")

# Check Redis for presence data
r = get_redis()
presence = r.hgetall("doc:1:presence")
print(f"\n📍 Presence data in Redis for doc 1:")
for user_id, data in presence.items():
    info = json.loads(data)
    print(f"  User {user_id}: {info['name']} (cursor: {info['cursor']}, color: {info['color']})")

print(f"\n💡 Redis stores presence for cross-server awareness.")
print(f"   In a multi-server setup, server B can read server A's user presence.")

Bob sees Alice's cursor at position 50

📍 Presence data in Redis for doc 1:
  User 1: Alice Johnson (cursor: 50, color: #3B82F6)
  User 2: Bob Smith (cursor: 0, color: #EF4444)

💡 Redis stores presence for cross-server awareness.
   In a multi-server setup, server B can read server A's user presence.


In [9]:
# Step 5: Bob disconnects — Alice gets notified

await bob.disconnect()

alice_msg = await alice.recv_message(timeout=1)
if alice_msg and alice_msg["type"] == "user_left":
    print(f"Alice received: user_left (user_id={alice_msg['user_id']})")

# Check Redis — Bob should be removed
presence = r.hgetall("doc:1:presence")
print(f"\nPresence after Bob left:")
for user_id, data in presence.items():
    info = json.loads(data)
    print(f"  User {user_id}: {info['name']}")

  [Bob] Disconnected
Alice received: user_left (user_id=2)

Presence after Bob left:
  User 1: Alice Johnson


## 🔐 Role-Based Permissions

Not every collaborator should be able to edit. Google Docs has three roles:

| Role | Can view | Can edit | Can share |
|------|----------|----------|-----------|
| **Owner** | ✅ | ✅ | ✅ |
| **Editor** | ✅ | ✅ | ❌ |
| **Viewer** | ✅ | ❌ | ❌ |

The server must enforce this **server-side** — never trust the client.
In our seed data, Charlie (user 3) is a **viewer** on document 3. Let's
verify that when Charlie tries to edit, the server rejects the operation.

In [10]:
# Charlie (user 3) is a VIEWER on document 3 — edits should be rejected

charlie = DocClient(user_id=3, name="Charlie")
await charlie.connect()
await charlie.join_doc(3)  # Shared Shopping List — Charlie's role is 'viewer'

# Try to edit — server should respond with an error
await charlie.ws.send(json.dumps({
    "type": "edit",
    "document_id": 3,
    "op_type": "insert",
    "position": 0,
    "content": "evil edit",
}))
resp = await charlie.recv_message(timeout=1)
print(f"Server response to Charlie's edit: {resp}")

if resp and resp.get("type") == "error":
    print(f"\n✅ Viewer correctly blocked from editing.")
else:
    print(f"\n❌ Uh oh — viewer was allowed to edit!")

await charlie.disconnect()

print(f"\n💡 Permissions are checked on EVERY edit on the server.")
print(f"   The client UI can disable the editor for viewers, but the server")
print(f"   is the true source of security — it never trusts client messages.")


  [Charlie] Connected: {'type': 'connected', 'user_id': 3}
  [Charlie] Joined doc 3 (version 0)
  [Charlie] Document: ''
  [Charlie] No other editors currently
Server response to Charlie's edit: {'type': 'error', 'message': "Permission denied: your role is 'viewer' (read-only)."}

✅ Viewer correctly blocked from editing.
  [Charlie] Disconnected

💡 Permissions are checked on EVERY edit on the server.
   The client UI can disable the editor for viewers, but the server
   is the true source of security — it never trusts client messages.


## 🏗️ Scaling WebSocket Connections

With millions of users, one server can't handle all connections. The system design solution:

```
┌──────────┐     ┌──────────┐     ┌──────────┐
│ Client A  │     │ Client B  │     │ Client C  │
└─────┬────┘     └─────┬────┘     └─────┬────┘
      │               │               │
      ▼               ▼               ▼
┌───────────┐   ┌───────────┐   ┌───────────┐
│ Doc Server│   │ Doc Server│   │ Doc Server│
│    #1     │   │    #2     │   │    #3     │
└─────┬─────┘   └─────┬─────┘   └─────┬─────┘
      │               │               │
      └───────────────┼───────────────┘
                      │
             ┌────────▼────────┐
             │ Consistent Hash │
             │   Ring (ZK)     │
             └─────────────────┘
```

**Key points:**
- **Consistent hashing** routes all editors of the same document to the same server
- **OT requires** all editors on one server (central authority for operation ordering)
- When a server fails, its documents are redistributed to other servers
- Client reconnects to the correct server via hash ring lookup

In [11]:
# Simulating consistent hashing for document routing
import hashlib

def consistent_hash(key, num_servers):
    """Simple consistent hash: maps a key to a server index."""
    hash_val = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
    return hash_val % num_servers

num_servers = 5
server_names = [f"doc-server-{i}" for i in range(num_servers)]

print(f"📊 Document → Server Routing ({num_servers} servers)")
print("=" * 50)

# Route 20 documents
server_load = {i: [] for i in range(num_servers)}
for doc_id in range(1, 21):
    server_idx = consistent_hash(f"doc:{doc_id}", num_servers)
    server_load[server_idx].append(doc_id)
    print(f"  Doc {doc_id:>2} → {server_names[server_idx]}")

print(f"\n📊 Server Load Distribution:")
for idx, docs in server_load.items():
    bar = "█" * len(docs)
    print(f"  {server_names[idx]}: {bar} ({len(docs)} docs)")

print(f"\n💡 All editors of the same document go to the same server.")
print(f"   This is critical for OT — the server needs all ops for ordering.")

📊 Document → Server Routing (5 servers)
  Doc  1 → doc-server-0
  Doc  2 → doc-server-4
  Doc  3 → doc-server-4
  Doc  4 → doc-server-3
  Doc  5 → doc-server-1
  Doc  6 → doc-server-2
  Doc  7 → doc-server-0
  Doc  8 → doc-server-4
  Doc  9 → doc-server-1
  Doc 10 → doc-server-2
  Doc 11 → doc-server-2
  Doc 12 → doc-server-3
  Doc 13 → doc-server-3
  Doc 14 → doc-server-2
  Doc 15 → doc-server-2
  Doc 16 → doc-server-4
  Doc 17 → doc-server-4
  Doc 18 → doc-server-4
  Doc 19 → doc-server-0
  Doc 20 → doc-server-4

📊 Server Load Distribution:
  doc-server-0: ███ (3 docs)
  doc-server-1: ██ (2 docs)
  doc-server-2: █████ (5 docs)
  doc-server-3: ███ (3 docs)
  doc-server-4: ███████ (7 docs)

💡 All editors of the same document go to the same server.
   This is critical for OT — the server needs all ops for ordering.


In [12]:
# Clean up connections
await alice.disconnect()

  [Alice] Disconnected


## 🧹 Cleanup

In [13]:
# Clean up Redis presence keys
r = get_redis()
for key in r.keys("doc:*:presence"):
    r.delete(key)
print("🧹 Cleaned up Redis presence keys")

🧹 Cleaned up Redis presence keys


## 📚 Summary

### Key Takeaways

1. **WebSockets are essential** — bi-directional, low-latency, persistent connections for real-time editing
2. **Connection lifecycle**: connect → join doc → receive state → edit loop → disconnect
3. **Cursor presence** uses Redis hashes — ephemeral data cleared on disconnect
4. **Broadcast pattern**: server receives an edit, ACKs the sender, broadcasts to all other editors
5. **Scaling**: consistent hashing routes all editors of a document to the same server

### For System Design Interviews

- Always mention **WebSockets** for real-time collaborative features
- Discuss the **single-server-per-document** constraint from OT
- Know how **consistent hashing** distributes documents across servers
- Mention **Redis pub/sub or presence** for cross-server awareness

### Next Up

In **Notebook 4**, we'll explore **document versioning and history** — how snapshots, compaction, and version restore work.